# KuaiRand Two-Tower Gold Build on Colab

Run this notebook on a **High-RAM CPU** Colab runtime. GPU is not needed.

Expected Drive input:

```text
MyDrive/recsys/data/silver/kuairand/interactions/
MyDrive/recsys/data/silver/kuairand/users/
MyDrive/recsys/data/silver/kuairand/videos_basic/
```

Output will be copied to:

```text
MyDrive/recsys/data/gold/two_tower/v1_colab/
```

This notebook is standalone: it writes the required project code into the Colab runtime and does not clone GitHub.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Install PySpark

This installs only Spark dependencies. It should be much lighter than installing the full project stack.

In [ ]:
%%bash
set -e
pip install -q pyspark==3.5.6 py4j==0.10.9.7


## Configure Paths and Runtime

Use local Colab disk for Spark work and final Parquet writing, then copy successful artifacts back to Drive.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
DRIVE_SILVER_DIR = DRIVE_ROOT / 'data/silver/kuairand'
DRIVE_OUTPUT_DIR = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'

LOCAL_ROOT = Path('/content/recsys_work')
LOCAL_SILVER_DIR = LOCAL_ROOT / 'data/silver/kuairand'
LOCAL_OUTPUT_DIR = LOCAL_ROOT / 'data/gold/two_tower/v1_colab'
SPARK_TMP = Path('/content/spark-tmp')


def total_ram_gb() -> float:
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                return int(line.split()[1]) / 1024**2
    return 12.0

ram_gb = total_ram_gb()
# Leave memory for Python, OS, Drive FUSE, and Spark overhead.
# Standard Colab (~12.7GB) gets 8g; High-RAM usually gets 24g+.
driver_memory_gb = max(6, min(24, int(ram_gb * 0.65)))
local_threads = 2 if ram_gb < 20 else 4

os.environ['SPARK_MASTER'] = f'local[{local_threads}]'
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--driver-memory {driver_memory_gb}g pyspark-shell'
os.environ['SPARK_LOCAL_DIR'] = str(SPARK_TMP)
os.environ['SPARK_LOCAL_DIRS'] = str(SPARK_TMP)
os.environ['LOCAL_DIRS'] = str(SPARK_TMP)
# Avoid trying to cache the large point-in-time example table in RAM on standard Colab.
os.environ['TWO_TOWER_STORAGE_LEVEL'] = 'DISK_ONLY'

for p in [LOCAL_ROOT, LOCAL_SILVER_DIR, LOCAL_OUTPUT_DIR.parent, SPARK_TMP]:
    p.mkdir(parents=True, exist_ok=True)

print('Detected RAM GB:', round(ram_gb, 2))
print('Spark master:', os.environ['SPARK_MASTER'])
print('Spark submit args:', os.environ['PYSPARK_SUBMIT_ARGS'])
print('Storage level:', os.environ['TWO_TOWER_STORAGE_LEVEL'])
print('Drive silver input:', DRIVE_SILVER_DIR)
print('Local silver input:', LOCAL_SILVER_DIR)
print('Local gold output:', LOCAL_OUTPUT_DIR)
print('Drive gold output:', DRIVE_OUTPUT_DIR)
print('Disk free local:', round(shutil.disk_usage('/content').free / 1024**3, 2), 'GB')


## Validate and Copy Silver Input

The build needs only these Silver tables: `interactions`, `users`, and `videos_basic`.

In [ ]:
required_tables = ['interactions', 'users', 'videos_basic']
missing = [name for name in required_tables if not (DRIVE_SILVER_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing Drive Silver tables: {missing}. Expected under {DRIVE_SILVER_DIR}')

if LOCAL_SILVER_DIR.exists():
    shutil.rmtree(LOCAL_SILVER_DIR)
LOCAL_SILVER_DIR.mkdir(parents=True, exist_ok=True)

for name in required_tables:
    src = DRIVE_SILVER_DIR / name
    dst = LOCAL_SILVER_DIR / name
    print(f'Copying {src} -> {dst}')
    shutil.copytree(src, dst)

for name in required_tables:
    size_gb = sum(p.stat().st_size for p in (LOCAL_SILVER_DIR / name).rglob('*') if p.is_file()) / 1024**3
    print(f'{name}: {size_gb:.2f} GB')


## Write Pipeline Code into Runtime

This cell materializes the same optimized pipeline code used in the repo commit `c5406b1` era.

In [ ]:
import textwrap

PACKAGE_ROOT = LOCAL_ROOT / 'recommender'
(PACKAGE_ROOT / 'gold').mkdir(parents=True, exist_ok=True)
(PACKAGE_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PACKAGE_ROOT / '__init__.py').write_text('')
(PACKAGE_ROOT / 'gold' / '__init__.py').write_text('')
(PACKAGE_ROOT / 'data' / '__init__.py').write_text('')

(PACKAGE_ROOT / 'spark.py').write_text('from __future__ import annotations\n\nimport os\nimport sys\nfrom pathlib import Path\n\nfrom py4j.protocol import Py4JError, Py4JNetworkError\nfrom pyspark import SparkContext\nfrom pyspark.sql import SparkSession\n\n\ndef _project_root() -> Path:\n    return Path(__file__).resolve().parents[1]\n\n\ndef _clear_spark_state() -> None:\n    SparkSession._instantiatedSession = None\n    SparkSession._activeSession = None\n    SparkContext._active_spark_context = None\n    SparkContext._gateway = None\n    SparkContext._jvm = None\n\n\ndef _session_is_healthy(session: SparkSession | None) -> bool:\n    if session is None:\n        return False\n\n    try:\n        session.range(1).count()\n        return True\n    except (ConnectionRefusedError, Py4JError, Py4JNetworkError):\n        return False\n\n\ndef get_spark(app_name: str = "recommender", reset: bool = False) -> SparkSession:\n    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")\n    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 4g pyspark-shell")\n    os.environ["PYSPARK_PYTHON"] = sys.executable\n    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable\n    spark_local_dir = Path(os.getenv("SPARK_LOCAL_DIR", _project_root() / "data/spark-tmp")).resolve()\n    spark_local_dir.mkdir(parents=True, exist_ok=True)\n    os.environ["SPARK_LOCAL_DIRS"] = str(spark_local_dir)\n    os.environ["LOCAL_DIRS"] = str(spark_local_dir)\n\n    if reset:\n        _clear_spark_state()\n\n    existing = SparkSession._activeSession or SparkSession._instantiatedSession\n    if _session_is_healthy(existing):\n        return existing\n    if existing is not None:\n        try:\n            existing.stop()\n        except (ConnectionRefusedError, Py4JError, Py4JNetworkError):\n            pass\n        _clear_spark_state()\n\n    spark_master = os.getenv("SPARK_MASTER", "local[*]")\n\n    builder = (\n        SparkSession.builder.master(spark_master)\n        .appName(app_name)\n        .config("spark.sql.session.timeZone", "UTC")\n        .config("spark.sql.shuffle.partitions", "8")\n        .config("spark.driver.bindAddress", "127.0.0.1")\n        .config("spark.driver.host", "127.0.0.1")\n        .config("spark.local.dir", str(spark_local_dir))\n        .config("spark.sql.execution.arrow.pyspark.enabled", "true")\n        .config("spark.pyspark.python", sys.executable)\n        .config("spark.pyspark.driver.python", sys.executable)\n    )\n\n    try:\n        session = builder.getOrCreate()\n    except (ConnectionRefusedError, Py4JError, Py4JNetworkError):\n        _clear_spark_state()\n        session = builder.getOrCreate()\n\n    if not _session_is_healthy(session):\n        _clear_spark_state()\n        session = builder.getOrCreate()\n\n    return session\n')
(PACKAGE_ROOT / 'gold' / 'als_baseline.py').write_text('from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nimport subprocess\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Iterable\n\nfrom pyspark.ml.recommendation import ALS\nfrom pyspark.sql import DataFrame, SparkSession, Window\nfrom pyspark.sql import functions as F\n\nfrom recommender.spark import get_spark\n\n\nDEFAULT_WEIGHTS = {\n    "watch_ratio": 0.5,\n    "long_view": 1.0,\n    "is_like": 1.5,\n    "is_comment": 1.5,\n    "is_forward": 1.5,\n    "is_follow": 2.0,\n}\nFORMULA_VERSION = "implicit_strength_v1"\n\n\n@dataclass(frozen=True)\nclass AlsParams:\n    rank: int = 64\n    reg_param: float = 0.05\n    alpha: float = 20.0\n    max_iter: int = 8\n\n\ndef parse_float_list(value: str) -> list[float]:\n    return [float(v.strip()) for v in value.split(",") if v.strip()]\n\n\ndef parse_int_list(value: str) -> list[int]:\n    return [int(v.strip()) for v in value.split(",") if v.strip()]\n\n\ndef git_commit() -> str | None:\n    try:\n        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()\n    except Exception:\n        return None\n\n\ndef read_interactions(spark: SparkSession, silver_dir: Path) -> DataFrame:\n    path = silver_dir / "interactions"\n    if not path.exists():\n        raise FileNotFoundError(f"Missing silver interactions table: {path}")\n    df = spark.read.parquet(str(path))\n    required = {\n        "user_id",\n        "video_id",\n        "event_ts",\n        "time_ms",\n        "watch_ratio",\n        "long_view",\n        "is_like",\n        "is_comment",\n        "is_forward",\n        "is_follow",\n    }\n    missing = sorted(required - set(df.columns))\n    if missing:\n        raise ValueError(f"Silver interactions is missing required columns: {missing}")\n    return df\n\n\ndef split_events(events: DataFrame, train_q: float = 0.70, validation_q: float = 0.85) -> tuple[DataFrame, dict]:\n    cut_train, cut_validation = events.approxQuantile("time_ms", [train_q, validation_q], 0.001)\n    split = (\n        events.where(F.col("event_ts").isNotNull())\n        .withColumn(\n            "split",\n            F.when(F.col("time_ms") < F.lit(cut_train), F.lit("train"))\n            .when(F.col("time_ms") < F.lit(cut_validation), F.lit("validation"))\n            .otherwise(F.lit("test")),\n        )\n    )\n    split_stats = {\n        "train_cutoff_time_ms": int(cut_train),\n        "validation_cutoff_time_ms": int(cut_validation),\n        "train_cutoff_ts": split.where(F.col("time_ms") >= cut_train).agg(F.min("event_ts")).first()[0].isoformat(),\n        "validation_cutoff_ts": split.where(F.col("time_ms") >= cut_validation).agg(F.min("event_ts")).first()[0].isoformat(),\n    }\n    return split, split_stats\n\n\ndef add_event_strength(events: DataFrame, weights: dict[str, float]) -> DataFrame:\n    clipped_watch_ratio = F.least(F.greatest(F.coalesce(F.col("watch_ratio"), F.lit(0.0)), F.lit(0.0)), F.lit(3.0))\n    strength = F.lit(weights["watch_ratio"]) * clipped_watch_ratio\n    for col in ["long_view", "is_like", "is_comment", "is_forward", "is_follow"]:\n        strength = strength + F.lit(weights[col]) * F.coalesce(F.col(col), F.lit(0)).cast("double")\n    return events.withColumn("event_strength", F.greatest(strength, F.lit(0.0)))\n\n\ndef strong_relevance_expr() -> F.Column:\n    return (\n        (F.coalesce(F.col("long_view"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_like"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_comment"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_forward"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_follow"), F.lit(0)) == 1)\n    )\n\n\ndef graded_relevance_expr(weights: dict[str, float]) -> F.Column:\n    return (\n        F.lit(weights["long_view"]) * F.coalesce(F.col("long_view"), F.lit(0)).cast("double")\n        + F.lit(weights["is_like"]) * F.coalesce(F.col("is_like"), F.lit(0)).cast("double")\n        + F.lit(weights["is_comment"]) * F.coalesce(F.col("is_comment"), F.lit(0)).cast("double")\n        + F.lit(weights["is_forward"]) * F.coalesce(F.col("is_forward"), F.lit(0)).cast("double")\n        + F.lit(weights["is_follow"]) * F.coalesce(F.col("is_follow"), F.lit(0)).cast("double")\n    )\n\n\ndef split_summary(events: DataFrame) -> list[dict]:\n    rows = (\n        events.groupBy("split")\n        .agg(\n            F.count("*").alias("events"),\n            F.countDistinct("user_id").alias("users"),\n            F.countDistinct("video_id").alias("items"),\n            F.min("event_ts").alias("min_ts"),\n            F.max("event_ts").alias("max_ts"),\n        )\n        .orderBy("split")\n        .collect()\n    )\n    return [\n        {\n            "split": r["split"],\n            "events": int(r["events"]),\n            "users": int(r["users"]),\n            "items": int(r["items"]),\n            "min_ts": r["min_ts"].isoformat() if r["min_ts"] else None,\n            "max_ts": r["max_ts"].isoformat() if r["max_ts"] else None,\n        }\n        for r in rows\n    ]\n\n\ndef build_train_interactions(train_events: DataFrame) -> DataFrame:\n    return (\n        train_events.groupBy("user_id", "video_id")\n        .agg(F.log1p(F.sum("event_strength")).alias("interaction_strength"))\n        .where(F.col("interaction_strength") > 0)\n    )\n\n\ndef build_mappings(train_interactions: DataFrame) -> tuple[DataFrame, DataFrame]:\n    user_window = Window.orderBy("user_id")\n    item_window = Window.orderBy("video_id")\n    users = train_interactions.select("user_id").distinct().withColumn("user_idx", F.row_number().over(user_window) - 1)\n    items = train_interactions.select("video_id").distinct().withColumn("video_idx", F.row_number().over(item_window) - 1)\n    return users, items\n\n\ndef apply_mappings(interactions: DataFrame, users: DataFrame, items: DataFrame) -> DataFrame:\n    return (\n        interactions.join(F.broadcast(users), on="user_id", how="inner")\n        .join(items, on="video_id", how="inner")\n        .select("user_id", "video_id", "user_idx", "video_idx", "interaction_strength")\n    )\n\n\ndef build_relevance(events: DataFrame, users: DataFrame, items: DataFrame, split: str, weights: dict[str, float]) -> DataFrame:\n    relevant = (\n        events.where(F.col("split") == split)\n        .where(strong_relevance_expr())\n        .withColumn("graded_relevance_event", graded_relevance_expr(weights))\n        .groupBy("user_id", "video_id")\n        .agg(\n            F.lit(1).alias("relevant"),\n            F.max("graded_relevance_event").alias("graded_relevance"),\n            F.count("*").alias("relevant_events"),\n        )\n    )\n    return (\n        relevant.join(F.broadcast(users), on="user_id", how="left")\n        .join(items, on="video_id", how="left")\n        .withColumn("is_warm_user", F.col("user_idx").isNotNull())\n        .withColumn("is_warm_item", F.col("video_idx").isNotNull())\n    )\n\n\ndef cold_start_report(relevance: DataFrame) -> dict:\n    total = relevance.count()\n    if total == 0:\n        return {"rows": 0, "warm_user_rate": 0.0, "warm_item_rate": 0.0, "cold_user_rate": 0.0, "cold_item_rate": 0.0}\n    row = relevance.agg(\n        F.avg(F.col("is_warm_user").cast("double")).alias("warm_user_rate"),\n        F.avg(F.col("is_warm_item").cast("double")).alias("warm_item_rate"),\n        F.countDistinct("user_id").alias("users"),\n        F.countDistinct("video_id").alias("items"),\n    ).first()\n    return {\n        "rows": int(total),\n        "users": int(row["users"]),\n        "items": int(row["items"]),\n        "warm_user_rate": float(row["warm_user_rate"] or 0.0),\n        "warm_item_rate": float(row["warm_item_rate"] or 0.0),\n        "cold_user_rate": float(1.0 - (row["warm_user_rate"] or 0.0)),\n        "cold_item_rate": float(1.0 - (row["warm_item_rate"] or 0.0)),\n    }\n\n\ndef ranking_metrics(recommendations: DataFrame, relevance: DataFrame, k: int = 10) -> dict:\n    rel = relevance.where(F.col("is_warm_user") & F.col("is_warm_item")).select(\n        "user_idx", "video_idx", "graded_relevance"\n    )\n    rel_by_user = rel.groupBy("user_idx").agg(F.countDistinct("video_idx").alias("num_relevant"))\n    evaluated_users = rel_by_user.count()\n    if evaluated_users == 0:\n        return {"evaluated_users": 0, f"recall@{k}": 0.0, f"ndcg@{k}": 0.0, f"hitrate@{k}": 0.0}\n\n    joined = (\n        recommendations.where(F.col("rank") <= k)\n        .join(rel, on=["user_idx", "video_idx"], how="left")\n        .withColumn("is_hit", F.col("graded_relevance").isNotNull().cast("int"))\n        .withColumn("gain", F.coalesce(F.col("graded_relevance"), F.lit(0.0)))\n        .withColumn("dcg_term", F.col("gain") / (F.log2(F.col("rank") + F.lit(1.0))))\n    )\n    per_user_dcg = joined.groupBy("user_idx").agg(\n        F.sum("is_hit").alias("hits"),\n        F.sum("dcg_term").alias("dcg"),\n        F.countDistinct("video_idx").alias("recommended_items"),\n    )\n\n    ideal_window = Window.partitionBy("user_idx").orderBy(F.desc("graded_relevance"), F.asc("video_idx"))\n    ideal = (\n        rel.withColumn("ideal_rank", F.row_number().over(ideal_window))\n        .where(F.col("ideal_rank") <= k)\n        .withColumn("idcg_term", F.col("graded_relevance") / F.log2(F.col("ideal_rank") + F.lit(1.0)))\n        .groupBy("user_idx")\n        .agg(F.sum("idcg_term").alias("idcg"))\n    )\n    metrics = (\n        rel_by_user.join(per_user_dcg, on="user_idx", how="left")\n        .join(ideal, on="user_idx", how="left")\n        .fillna({"hits": 0.0, "dcg": 0.0, "recommended_items": 0.0, "idcg": 0.0})\n        .withColumn("recall", F.col("hits") / F.least(F.col("num_relevant"), F.lit(k)))\n        .withColumn("hitrate", (F.col("hits") > 0).cast("double"))\n        .withColumn("ndcg", F.when(F.col("idcg") > 0, F.col("dcg") / F.col("idcg")).otherwise(F.lit(0.0)))\n    )\n    row = metrics.agg(\n        F.count("*").alias("evaluated_users"),\n        F.avg("recall").alias("recall"),\n        F.avg("ndcg").alias("ndcg"),\n        F.avg("hitrate").alias("hitrate"),\n        F.avg("recommended_items").alias("avg_recommended_items"),\n    ).first()\n    return {\n        "evaluated_users": int(row["evaluated_users"]),\n        f"recall@{k}": float(row["recall"] or 0.0),\n        f"ndcg@{k}": float(row["ndcg"] or 0.0),\n        f"hitrate@{k}": float(row["hitrate"] or 0.0),\n        "avg_recommended_items": float(row["avg_recommended_items"] or 0.0),\n    }\n\n\ndef popularity_recommendations(\n    train_interactions: DataFrame,\n    users_to_eval: DataFrame,\n    train_history: DataFrame,\n    k: int,\n    candidate_multiplier: int = 300,\n) -> DataFrame:\n    top_n = max(k * candidate_multiplier, 1000)\n    popular = (\n        train_interactions.groupBy("video_idx")\n        .agg(F.sum("interaction_strength").alias("popularity_score"))\n        .orderBy(F.desc("popularity_score"), F.asc("video_idx"))\n        .limit(top_n)\n    )\n    candidates = users_to_eval.select("user_idx").distinct().crossJoin(F.broadcast(popular))\n    filtered = candidates.join(train_history.select("user_idx", "video_idx"), on=["user_idx", "video_idx"], how="left_anti")\n    window = Window.partitionBy("user_idx").orderBy(F.desc("popularity_score"), F.asc("video_idx"))\n    return filtered.withColumn("rank", F.row_number().over(window)).where(F.col("rank") <= k)\n\n\ndef als_recommendations(model, users_to_eval: DataFrame, train_history: DataFrame, k: int, over_generate: int = 300) -> DataFrame:\n    raw = model.recommendForUserSubset(users_to_eval.select("user_idx").distinct(), max(k, over_generate))\n    exploded = raw.select("user_idx", F.posexplode("recommendations").alias("pos", "rec")).select(\n        "user_idx",\n        F.col("rec.video_idx").alias("video_idx"),\n        F.col("rec.rating").alias("score"),\n    )\n    filtered = exploded.join(train_history.select("user_idx", "video_idx"), on=["user_idx", "video_idx"], how="left_anti")\n    window = Window.partitionBy("user_idx").orderBy(F.desc("score"), F.asc("video_idx"))\n    return filtered.withColumn("rank", F.row_number().over(window)).where(F.col("rank") <= k)\n\n\ndef train_als(train: DataFrame, params: AlsParams):\n    als = ALS(\n        userCol="user_idx",\n        itemCol="video_idx",\n        ratingCol="interaction_strength",\n        implicitPrefs=True,\n        coldStartStrategy="drop",\n        nonnegative=False,\n        rank=params.rank,\n        regParam=params.reg_param,\n        alpha=params.alpha,\n        maxIter=params.max_iter,\n        seed=42,\n    )\n    return als.fit(train.select("user_idx", "video_idx", "interaction_strength"))\n\n\ndef tune_strength_weights(\n    train_events: DataFrame,\n    validation_relevance: DataFrame,\n    users: DataFrame,\n    items: DataFrame,\n    trials: int,\n    k: int,\n) -> dict[str, float]:\n    if trials <= 0:\n        return DEFAULT_WEIGHTS\n\n    def evaluate(weights: dict[str, float]) -> float:\n        weighted = add_event_strength(train_events, weights)\n        train_interactions = apply_mappings(build_train_interactions(weighted), users, items).cache()\n        recs = popularity_recommendations(\n            train_interactions,\n            validation_relevance.where(F.col("is_warm_user")).select("user_idx").distinct(),\n            train_interactions.select("user_idx", "video_idx"),\n            k=k,\n        )\n        score = ranking_metrics(recs, validation_relevance, k=k)[f"ndcg@{k}"]\n        train_interactions.unpersist()\n        return score\n\n    try:\n        import optuna  # type: ignore\n\n        def objective(trial):\n            weights = {\n                "watch_ratio": trial.suggest_float("watch_ratio", 0.1, 1.0),\n                "long_view": trial.suggest_float("long_view", 0.5, 2.0),\n                "is_like": trial.suggest_float("is_like", 0.5, 3.0),\n                "is_comment": trial.suggest_float("is_comment", 0.5, 3.0),\n                "is_forward": trial.suggest_float("is_forward", 0.5, 3.0),\n                "is_follow": trial.suggest_float("is_follow", 1.0, 4.0),\n            }\n            return evaluate(weights)\n\n        study = optuna.create_study(direction="maximize")\n        study.optimize(objective, n_trials=trials)\n        return {k: float(v) for k, v in study.best_params.items()}\n    except Exception:\n        rng = random.Random(42)\n        best_weights = DEFAULT_WEIGHTS\n        best_score = evaluate(best_weights)\n        for _ in range(trials):\n            weights = {\n                "watch_ratio": rng.uniform(0.1, 1.0),\n                "long_view": rng.uniform(0.5, 2.0),\n                "is_like": rng.uniform(0.5, 3.0),\n                "is_comment": rng.uniform(0.5, 3.0),\n                "is_forward": rng.uniform(0.5, 3.0),\n                "is_follow": rng.uniform(1.0, 4.0),\n            }\n            score = evaluate(weights)\n            if score > best_score:\n                best_score = score\n                best_weights = weights\n        return best_weights\n\n\ndef write_parquet(df: DataFrame, path: Path, overwrite: bool) -> None:\n    mode = "overwrite" if overwrite else "errorifexists"\n    df.write.mode(mode).parquet(str(path))\n\n\ndef write_json(data: dict, path: Path) -> None:\n    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\\n")\n\n\ndef run_pipeline(args: argparse.Namespace) -> dict:\n    spark = get_spark("kuairand-als-baseline", reset=True)\n    output_dir = args.output_dir\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    raw_events = read_interactions(spark, args.silver_dir)\n    split, split_meta = split_events(raw_events)\n    split = split.cache()\n    base_summary = split_summary(split)\n\n    weighted_default = add_event_strength(split, DEFAULT_WEIGHTS)\n    train_events = weighted_default.where(F.col("split") == "train").cache()\n    train_interactions_original = build_train_interactions(train_events).cache()\n    user_mapping, item_mapping = build_mappings(train_interactions_original)\n    user_mapping = user_mapping.cache()\n    item_mapping = item_mapping.cache()\n\n    validation_relevance_default = build_relevance(weighted_default, user_mapping, item_mapping, "validation", DEFAULT_WEIGHTS).cache()\n    selected_weights = tune_strength_weights(\n        split.where(F.col("split") == "train"),\n        validation_relevance_default,\n        user_mapping,\n        item_mapping,\n        args.strength_tuning_trials,\n        args.k,\n    )\n    weighted = add_event_strength(split, selected_weights)\n    train_events = weighted.where(F.col("split") == "train").cache()\n    train_interactions_original = build_train_interactions(train_events).cache()\n    user_mapping, item_mapping = build_mappings(train_interactions_original)\n    user_mapping = user_mapping.cache()\n    item_mapping = item_mapping.cache()\n    train_interactions = apply_mappings(train_interactions_original, user_mapping, item_mapping).cache()\n\n    validation_relevance = build_relevance(weighted, user_mapping, item_mapping, "validation", selected_weights).cache()\n    test_relevance = build_relevance(weighted, user_mapping, item_mapping, "test", selected_weights).cache()\n    train_history = train_interactions.select("user_idx", "video_idx").cache()\n\n    write_parquet(train_interactions.select("user_idx", "video_idx", "interaction_strength", "user_id", "video_id"), output_dir / "train_interactions", args.overwrite)\n    write_parquet(validation_relevance, output_dir / "validation_relevance", args.overwrite)\n    write_parquet(test_relevance, output_dir / "test_relevance", args.overwrite)\n    write_parquet(user_mapping, output_dir / "user_mapping", args.overwrite)\n    write_parquet(item_mapping, output_dir / "item_mapping", args.overwrite)\n\n    validation_users = validation_relevance.where(F.col("is_warm_user")).select("user_idx").distinct()\n    test_users = test_relevance.where(F.col("is_warm_user")).select("user_idx").distinct()\n\n    pop_val_recs = popularity_recommendations(train_interactions, validation_users, train_history, args.k).cache()\n    pop_test_recs = popularity_recommendations(train_interactions, test_users, train_history, args.k).cache()\n    pop_validation = ranking_metrics(pop_val_recs, validation_relevance, k=args.k)\n    pop_test = ranking_metrics(pop_test_recs, test_relevance, k=args.k)\n\n    grid = [\n        AlsParams(rank=rank, reg_param=reg, alpha=alpha, max_iter=args.max_iter)\n        for rank, reg, alpha in itertools.product(args.ranks, args.reg_params, args.alphas)\n    ]\n    if args.max_grid_models:\n        grid = grid[: args.max_grid_models]\n\n    als_results = []\n    best = None\n    for params in grid:\n        model = train_als(train_interactions, params)\n        recs = als_recommendations(model, validation_users, train_history, args.k, over_generate=args.als_over_generate).cache()\n        metrics = ranking_metrics(recs, validation_relevance, k=args.k)\n        result = {"params": asdict(params), "validation": metrics}\n        als_results.append(result)\n        score = metrics[f"ndcg@{args.k}"]\n        if best is None or score > best["score"]:\n            best = {"score": score, "params": params}\n        recs.unpersist()\n\n    if best is None:\n        raise RuntimeError("ALS grid was empty")\n\n    selected_params: AlsParams = best["params"]\n    train_validation_events = weighted.where(F.col("split").isin("train", "validation")).cache()\n    final_original = build_train_interactions(train_validation_events).cache()\n    final_users, final_items = build_mappings(final_original)\n    final_users = final_users.cache()\n    final_items = final_items.cache()\n    final_train = apply_mappings(final_original, final_users, final_items).cache()\n    final_history = final_train.select("user_idx", "video_idx").cache()\n    final_test_relevance = build_relevance(weighted, final_users, final_items, "test", selected_weights).cache()\n    final_test_users = final_test_relevance.where(F.col("is_warm_user")).select("user_idx").distinct()\n\n    final_model = train_als(final_train, selected_params)\n    final_test_recs = als_recommendations(final_model, final_test_users, final_history, args.k, over_generate=args.als_over_generate).cache()\n    final_test_metrics = ranking_metrics(final_test_recs, final_test_relevance, k=args.k)\n\n    write_parquet(final_train.select("user_idx", "video_idx", "interaction_strength", "user_id", "video_id"), output_dir / "train_validation_interactions", args.overwrite)\n    write_parquet(final_users, output_dir / "final_user_mapping", args.overwrite)\n    write_parquet(final_items, output_dir / "final_item_mapping", args.overwrite)\n    write_parquet(final_model.userFactors.withColumnRenamed("id", "user_idx"), output_dir / "user_factors", args.overwrite)\n    write_parquet(final_model.itemFactors.withColumnRenamed("id", "video_idx"), output_dir / "item_factors", args.overwrite)\n\n    manifest = {\n        "gold_dataset_version": args.version,\n        "generated_at": datetime.now(timezone.utc).isoformat(),\n        "source_silver_path": str(args.silver_dir),\n        "output_path": str(output_dir),\n        "code_commit": git_commit(),\n        "interaction_strength": {\n            "formula_version": FORMULA_VERSION,\n            "formula": "log1p(sum(0.5*clip(watch_ratio,0,3)+1.0*long_view+1.5*like+1.5*comment+1.5*forward+2.0*follow)) with selected weights if tuning is enabled",\n            "weights": selected_weights,\n            "negative_feedback_policy": "is_hate is retained in silver but not encoded as negative ALS rating in v1",\n        },\n        "temporal_split": split_meta,\n        "split_summary": base_summary,\n        "mapping_statistics_train_only": {\n            "users": user_mapping.count(),\n            "items": item_mapping.count(),\n            "train_user_item_rows": train_interactions.count(),\n        },\n        "cold_start": {\n            "validation": cold_start_report(validation_relevance),\n            "test": cold_start_report(test_relevance),\n            "final_test_after_train_validation": cold_start_report(final_test_relevance),\n        },\n        "evaluation_protocol": {\n            "split": "event-level chronological split before user-item aggregation",\n            "k": args.k,\n            "train_history_filter": "recommendations exclude items seen in the corresponding training history",\n            "metrics": [f"ndcg@{args.k}", f"recall@{args.k}", f"hitrate@{args.k}"],\n            "relevance": "binary relevant if any future long_view/like/comment/forward/follow; graded relevance from positive engagement weights",\n            "cold_start": "ranking metrics evaluate warm users and warm items; cold rates are reported separately",\n        },\n        "popularity_baseline": {"validation": pop_validation, "test": pop_test},\n        "als_grid_results": als_results,\n        "selected_als_hyperparameters": asdict(selected_params),\n        "final_als_test_metrics": final_test_metrics,\n        "artifact_paths": {\n            "train_interactions": str(output_dir / "train_interactions"),\n            "validation_relevance": str(output_dir / "validation_relevance"),\n            "test_relevance": str(output_dir / "test_relevance"),\n            "user_mapping": str(output_dir / "user_mapping"),\n            "item_mapping": str(output_dir / "item_mapping"),\n            "user_factors": str(output_dir / "user_factors"),\n            "item_factors": str(output_dir / "item_factors"),\n        },\n    }\n    write_json(manifest, output_dir / "manifest.json")\n    write_json(\n        {\n            "popularity_validation": pop_validation,\n            "popularity_test": pop_test,\n            "als_grid_results": als_results,\n            "final_als_test": final_test_metrics,\n        },\n        output_dir / "evaluation_summary.json",\n    )\n    print(json.dumps(manifest, indent=2, sort_keys=True))\n    spark.stop()\n    return manifest\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Build KuaiRand ALS gold data and train popularity/Spark ALS baselines.",\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n    )\n    parser.add_argument("--silver-dir", type=Path, default=Path("data/silver/kuairand"))\n    parser.add_argument("--output-dir", type=Path, default=Path("data/gold/als/v1"))\n    parser.add_argument("--version", default="als_v1")\n    parser.add_argument("--k", type=int, default=10)\n    parser.add_argument("--ranks", type=parse_int_list, default=[32, 64])\n    parser.add_argument("--reg-params", type=parse_float_list, default=[0.05, 0.1])\n    parser.add_argument("--alphas", type=parse_float_list, default=[10.0, 20.0])\n    parser.add_argument("--max-iter", type=int, default=8)\n    parser.add_argument("--max-grid-models", type=int, default=0, help="Optional cap for local smoke runs; 0 means all grid combinations.")\n    parser.add_argument("--als-over-generate", type=int, default=300)\n    parser.add_argument("--strength-tuning-trials", type=int, default=0, help="Optional Optuna/random-search trials for interaction-strength weights.")\n    parser.add_argument("--overwrite", action="store_true")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    run_pipeline(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n')
(PACKAGE_ROOT / 'gold' / 'two_tower.py').write_text('from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Iterable\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import DataFrame, SparkSession, Window\nfrom pyspark.sql import functions as F\n\nfrom recommender.gold.als_baseline import git_commit, split_events, split_summary, write_json, write_parquet\nfrom recommender.spark import get_spark\n\n\nDATASET_VERSION = "two_tower_v1"\nFEATURE_CATALOG_VERSION = "two_tower_feature_catalog_v1"\nTARGET_DEFINITION_VERSION = "direct_feedback_v1"\nHISTORY_DEFINITION_VERSION = "user_chronological_30m_session_v1"\nUNKNOWN_TOKEN = "__UNKNOWN__"\n\nUSER_ID = "user_id"\nITEM_ID = "video_id"\n\nDIRECT_FEEDBACK_COLS = [\n    "long_view",\n    "is_like",\n    "is_comment",\n    "is_forward",\n    "is_follow",\n    "is_hate",\n    "is_click",\n    "is_profile_enter",\n]\n\nUSER_STATIC_NUMERIC = [\n    "is_lowactive_period",\n    "is_live_streamer",\n    "is_video_author",\n    "follow_user_num",\n    "fans_user_num",\n    "friend_user_num",\n    "register_days",\n]\nUSER_STATIC_CATEGORICAL = ["user_active_degree"]\nUSER_ANON_NUMERIC = [f"onehot_feat{i}" for i in range(18)]\n\nUSER_HISTORY_NUMERIC = [\n    "user_hist_events",\n    "user_hist_long_view_rate",\n    "user_hist_like_rate",\n    "user_hist_comment_rate",\n    "user_hist_forward_rate",\n    "user_hist_follow_rate",\n    "user_hist_hate_rate",\n    "user_hist_avg_watch_ratio",\n    "user_hist_avg_play_time_sec",\n    "session_event_index",\n    "session_elapsed_sec",\n    "session_prior_long_view_rate",\n    "session_prior_like_rate",\n    "session_prior_hate_rate",\n    "session_prior_avg_watch_ratio",\n    "session_vs_user_long_view_delta",\n    "session_vs_user_watch_ratio_delta",\n]\nCONTEXT_FEATURES = ["event_hour", "event_dayofweek", "tab", "is_rand"]\n\nITEM_STATIC_NUMERIC = [\n    "video_duration_sec",\n    "aspect_ratio",\n    "visible_status",\n    "music_type",\n    "upload_age_days_at_event",\n]\nITEM_STATIC_CATEGORICAL = ["video_type", "upload_type"]\nITEM_HISTORY_NUMERIC = [\n    "item_hist_events",\n    "item_hist_long_view_rate",\n    "item_hist_like_rate",\n    "item_hist_hate_rate",\n    "item_hist_avg_watch_ratio",\n]\n\nTARGET_COLS = [\n    "target_class",\n    "is_positive",\n    "is_observed_negative",\n    "is_ambiguous",\n    "engagement_strength",\n    "watch_ratio_clipped",\n    *DIRECT_FEEDBACK_COLS,\n    "play_time_ms",\n    "duration_ms",\n    "watch_ratio",\n]\n\n\ndef storage_level_from_env() -> StorageLevel:\n    raw = os.getenv("TWO_TOWER_STORAGE_LEVEL", "DISK_ONLY").upper()\n    choices = {\n        "DISK_ONLY": StorageLevel.DISK_ONLY,\n        "MEMORY_ONLY": StorageLevel.MEMORY_ONLY,\n        "MEMORY_AND_DISK": StorageLevel.MEMORY_AND_DISK,\n    }\n    if raw not in choices:\n        raise ValueError(f"Unknown TWO_TOWER_STORAGE_LEVEL={raw}. Use one of {sorted(choices)}")\n    return choices[raw]\n\n\ndef read_silver_tables(spark: SparkSession, silver_dir: Path) -> dict[str, DataFrame]:\n    required = ["interactions", "users", "videos_basic"]\n    tables = {}\n    for name in required:\n        path = silver_dir / name\n        if not path.exists():\n            raise FileNotFoundError(f"Missing silver table: {path}")\n        tables[name] = spark.read.parquet(str(path))\n    return tables\n\n\ndef _require_columns(df: DataFrame, required: Iterable[str], table_name: str) -> None:\n    missing = sorted(set(required) - set(df.columns))\n    if missing:\n        raise ValueError(f"{table_name} is missing required columns: {missing}")\n\n\ndef validate_silver_schema(tables: dict[str, DataFrame]) -> None:\n    _require_columns(\n        tables["interactions"],\n        [\n            "event_id",\n            "user_id",\n            "video_id",\n            "event_ts",\n            "time_ms",\n            "event_hour",\n            "watch_ratio",\n            "play_time_ms",\n            "duration_ms",\n            "source_table",\n            *DIRECT_FEEDBACK_COLS,\n            "tab",\n            "is_rand",\n        ],\n        "silver interactions",\n    )\n    _require_columns(tables["users"], ["user_id", *USER_STATIC_CATEGORICAL], "silver users")\n    _require_columns(\n        tables["videos_basic"],\n        ["video_id", "video_type", "upload_type", "upload_date", "video_duration_sec", "aspect_ratio"],\n        "silver videos_basic",\n    )\n\n\ndef add_targets(events: DataFrame, weak_watch_ratio_threshold: float = 0.20) -> DataFrame:\n    positive_expr = (\n        (F.coalesce(F.col("long_view"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_like"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_comment"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_forward"), F.lit(0)) == 1)\n        | (F.coalesce(F.col("is_follow"), F.lit(0)) == 1)\n    )\n    watch_ratio_clipped = F.least(F.greatest(F.coalesce(F.col("watch_ratio"), F.lit(0.0)), F.lit(0.0)), F.lit(3.0))\n    observed_negative_expr = (\n        (~positive_expr)\n        & (\n            (F.coalesce(F.col("is_hate"), F.lit(0)) == 1)\n            | (\n                (F.coalesce(F.col("is_click"), F.lit(0)) == 1)\n                & (watch_ratio_clipped <= F.lit(weak_watch_ratio_threshold))\n            )\n        )\n    )\n    engagement_strength = (\n        F.lit(0.5) * watch_ratio_clipped\n        + F.lit(1.0) * F.coalesce(F.col("long_view"), F.lit(0)).cast("double")\n        + F.lit(1.5) * F.coalesce(F.col("is_like"), F.lit(0)).cast("double")\n        + F.lit(1.5) * F.coalesce(F.col("is_comment"), F.lit(0)).cast("double")\n        + F.lit(1.5) * F.coalesce(F.col("is_forward"), F.lit(0)).cast("double")\n        + F.lit(2.0) * F.coalesce(F.col("is_follow"), F.lit(0)).cast("double")\n        - F.lit(1.0) * F.coalesce(F.col("is_hate"), F.lit(0)).cast("double")\n    )\n    return (\n        events.withColumn("watch_ratio_clipped", watch_ratio_clipped)\n        .withColumn("is_positive", positive_expr.cast("int"))\n        .withColumn("is_observed_negative", observed_negative_expr.cast("int"))\n        .withColumn(\n            "target_class",\n            F.when(positive_expr, F.lit("STRONG_POSITIVE"))\n            .when(observed_negative_expr, F.lit("OBSERVED_NEGATIVE"))\n            .otherwise(F.lit("AMBIGUOUS_WEAK")),\n        )\n        .withColumn("is_ambiguous", (F.col("target_class") == "AMBIGUOUS_WEAK").cast("int"))\n        .withColumn("engagement_strength", engagement_strength)\n    )\n\n\ndef add_point_in_time_features(events: DataFrame) -> DataFrame:\n    user_order = Window.partitionBy("user_id").orderBy("time_ms", "event_id")\n    user_prev = user_order.rowsBetween(Window.unboundedPreceding, -1)\n    user_cur = user_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)\n    item_order = Window.partitionBy("video_id").orderBy("time_ms", "event_id")\n    item_prev = item_order.rowsBetween(Window.unboundedPreceding, -1)\n\n    with_session_seed = (\n        events.withColumn("prev_event_ts", F.lag("event_ts").over(user_order))\n        .withColumn("prev_video_id", F.lag("video_id").over(user_order))\n        .withColumn("user_event_index", F.row_number().over(user_order))\n        .withColumn("gap_sec", F.col("event_ts").cast("long") - F.col("prev_event_ts").cast("long"))\n        .withColumn(\n            "new_session_flag",\n            F.when(F.col("prev_event_ts").isNull() | (F.col("gap_sec") > 30 * 60), F.lit(1)).otherwise(F.lit(0)),\n        )\n        .withColumn("session_seq", F.sum("new_session_flag").over(user_cur))\n        .withColumn("session_id", F.concat_ws("_", F.col("user_id").cast("string"), F.col("session_seq").cast("string")))\n    )\n    session_order = Window.partitionBy("user_id", "session_seq").orderBy("time_ms", "event_id")\n    session_prev = session_order.rowsBetween(Window.unboundedPreceding, -1)\n    session_cur = session_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)\n\n    with_history = (\n        with_session_seed.withColumn("session_event_index", F.row_number().over(session_order))\n        .withColumn("session_start_ts", F.first("event_ts").over(session_cur))\n        .withColumn("session_elapsed_sec", F.col("event_ts").cast("long") - F.col("session_start_ts").cast("long"))\n        .withColumn("user_hist_events", F.count(F.lit(1)).over(user_prev))\n        .withColumn("user_hist_long_view_rate", F.avg(F.coalesce("long_view", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_like_rate", F.avg(F.coalesce("is_like", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_comment_rate", F.avg(F.coalesce("is_comment", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_forward_rate", F.avg(F.coalesce("is_forward", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_follow_rate", F.avg(F.coalesce("is_follow", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_hate_rate", F.avg(F.coalesce("is_hate", F.lit(0)).cast("double")).over(user_prev))\n        .withColumn("user_hist_avg_watch_ratio", F.avg("watch_ratio_clipped").over(user_prev))\n        .withColumn("user_hist_avg_play_time_sec", F.avg("play_time_sec").over(user_prev))\n        .withColumn("session_prior_long_view_rate", F.avg(F.coalesce("long_view", F.lit(0)).cast("double")).over(session_prev))\n        .withColumn("session_prior_like_rate", F.avg(F.coalesce("is_like", F.lit(0)).cast("double")).over(session_prev))\n        .withColumn("session_prior_hate_rate", F.avg(F.coalesce("is_hate", F.lit(0)).cast("double")).over(session_prev))\n        .withColumn("session_prior_avg_watch_ratio", F.avg("watch_ratio_clipped").over(session_prev))\n        .withColumn("item_hist_events", F.count(F.lit(1)).over(item_prev))\n        .withColumn("item_hist_long_view_rate", F.avg(F.coalesce("long_view", F.lit(0)).cast("double")).over(item_prev))\n        .withColumn("item_hist_like_rate", F.avg(F.coalesce("is_like", F.lit(0)).cast("double")).over(item_prev))\n        .withColumn("item_hist_hate_rate", F.avg(F.coalesce("is_hate", F.lit(0)).cast("double")).over(item_prev))\n        .withColumn("item_hist_avg_watch_ratio", F.avg("watch_ratio_clipped").over(item_prev))\n        .withColumn(\n            "session_vs_user_long_view_delta",\n            F.col("session_prior_long_view_rate") - F.col("user_hist_long_view_rate"),\n        )\n        .withColumn(\n            "session_vs_user_watch_ratio_delta",\n            F.col("session_prior_avg_watch_ratio") - F.col("user_hist_avg_watch_ratio"),\n        )\n        .withColumn("history_end_user_event_index", F.col("user_event_index") - F.lit(1))\n        .withColumn("history_end_session_event_index", F.col("session_event_index") - F.lit(1))\n    )\n    return with_history\n\n\ndef build_examples(\n    interactions: DataFrame,\n    users: DataFrame,\n    videos: DataFrame,\n    train_quantile: float,\n    validation_quantile: float,\n    weak_watch_ratio_threshold: float,\n) -> tuple[DataFrame, dict]:\n    events, split_meta = split_events(interactions, train_quantile, validation_quantile)\n    events = add_targets(events, weak_watch_ratio_threshold)\n    events = add_point_in_time_features(events)\n\n    users_keep = ["user_id", *USER_STATIC_CATEGORICAL, *USER_STATIC_NUMERIC, *USER_ANON_NUMERIC]\n    users_keep = [c for c in users_keep if c in users.columns]\n    videos_keep = [\n        "video_id",\n        "author_id",\n        *ITEM_STATIC_CATEGORICAL,\n        "upload_date",\n        "video_duration_is_missing",\n        "server_width_is_missing",\n        "server_height_is_missing",\n        "music_type_is_missing",\n        "tag_is_missing",\n        *[c for c in ITEM_STATIC_NUMERIC if c != "upload_age_days_at_event"],\n    ]\n    videos_keep = [c for c in videos_keep if c in videos.columns]\n\n    joined = events.join(F.broadcast(users.select(*users_keep)), on="user_id", how="left").join(\n        videos.select(*videos_keep), on="video_id", how="left"\n    )\n    joined = joined.withColumn("event_dayofweek", F.dayofweek("event_ts")).withColumn(\n        "upload_age_days_at_event", F.datediff(F.to_date("event_ts"), F.col("upload_date")).cast("double")\n    )\n    joined = joined.withColumn(\n        "example_id",\n        F.sha2(F.concat_ws("||", F.lit("example"), F.col("event_id"), F.col("time_ms").cast("string")), 256),\n    ).withColumn(\n        "context_id",\n        F.sha2(\n            F.concat_ws(\n                "||",\n                F.lit("context"),\n                F.col("user_id").cast("string"),\n                F.col("history_end_user_event_index").cast("string"),\n                F.col("session_id"),\n                F.col("history_end_session_event_index").cast("string"),\n            ),\n            256,\n        ),\n    )\n    joined = joined.withColumn("as_of_time", F.col("event_ts"))\n    return joined, split_meta\n\n\ndef split_counts(examples: DataFrame) -> list[dict]:\n    rows = (\n        examples.groupBy("split")\n        .agg(\n            F.count("*").alias("examples"),\n            F.countDistinct("user_id").alias("users"),\n            F.countDistinct("video_id").alias("items"),\n            F.sum((F.col("target_class") == "STRONG_POSITIVE").cast("long")).alias("strong_positive"),\n            F.sum((F.col("target_class") == "OBSERVED_NEGATIVE").cast("long")).alias("observed_negative"),\n            F.sum((F.col("target_class") == "AMBIGUOUS_WEAK").cast("long")).alias("ambiguous_weak"),\n        )\n        .orderBy("split")\n        .collect()\n    )\n    return [\n        {\n            "split": r["split"],\n            "examples": int(r["examples"]),\n            "users": int(r["users"]),\n            "items": int(r["items"]),\n            "target_counts": {\n                "STRONG_POSITIVE": int(r["strong_positive"] or 0),\n                "OBSERVED_NEGATIVE": int(r["observed_negative"] or 0),\n                "AMBIGUOUS_WEAK": int(r["ambiguous_weak"] or 0),\n            },\n        }\n        for r in rows\n    ]\n\n\ndef add_cold_start_flags(examples: DataFrame) -> DataFrame:\n    train_users = examples.where(F.col("split") == "train").select("user_id").distinct().withColumn("seen_user_in_train", F.lit(1))\n    train_items = examples.where(F.col("split") == "train").select("video_id").distinct().withColumn("seen_item_in_train", F.lit(1))\n    return (\n        examples.join(F.broadcast(train_users), on="user_id", how="left")\n        .join(train_items, on="video_id", how="left")\n        .withColumn("is_warm_user", F.coalesce(F.col("seen_user_in_train"), F.lit(0)).cast("boolean"))\n        .withColumn("is_warm_item", F.coalesce(F.col("seen_item_in_train"), F.lit(0)).cast("boolean"))\n        .withColumn("has_user_history", (F.col("history_end_user_event_index") > 0).cast("boolean"))\n        .withColumn("has_item_metadata", F.col("video_type").isNotNull())\n        .drop("seen_user_in_train", "seen_item_in_train")\n    )\n\n\ndef cold_start_stats(examples: DataFrame) -> dict[str, dict]:\n    rows = (\n        examples.groupBy("split")\n        .agg(\n            F.count("*").alias("examples"),\n            F.avg(F.col("is_warm_user").cast("double")).alias("warm_user_rate"),\n            F.avg(F.col("is_warm_item").cast("double")).alias("warm_item_rate"),\n            F.avg(F.col("has_user_history").cast("double")).alias("has_user_history_rate"),\n            F.avg(F.col("has_item_metadata").cast("double")).alias("has_item_metadata_rate"),\n        )\n        .collect()\n    )\n    stats = {split: {"examples": 0} for split in ["train", "validation", "test"]}\n    for row in rows:\n        stats[row["split"]] = {\n            "examples": int(row["examples"] or 0),\n            "warm_user_rate": float(row["warm_user_rate"] or 0.0),\n            "warm_item_rate": float(row["warm_item_rate"] or 0.0),\n            "cold_user_rate": float(1.0 - (row["warm_user_rate"] or 0.0)),\n            "cold_item_rate": float(1.0 - (row["warm_item_rate"] or 0.0)),\n            "has_user_history_rate": float(row["has_user_history_rate"] or 0.0),\n            "has_item_metadata_rate": float(row["has_item_metadata_rate"] or 0.0),\n        }\n    return stats\n\n\ndef build_vocabulary(df: DataFrame, column: str) -> DataFrame:\n    window = Window.orderBy(column)\n    return (\n        df.where(F.col(column).isNotNull())\n        .select(F.col(column).cast("string").alias(column))\n        .distinct()\n        .withColumn(f"{column}_idx", F.row_number().over(window))\n    )\n\n\ndef vocabulary_stats(train_examples: DataFrame, categorical_cols: list[str]) -> dict:\n    existing = [c for c in categorical_cols if c in train_examples.columns]\n    if not existing:\n        return {}\n\n    row = train_examples.agg(*[F.countDistinct(F.col(c)).alias(c) for c in existing]).first()\n    return {\n        col: {"fit_split": "train", "cardinality_excluding_oov": int(row[col] or 0)}\n        for col in existing\n    }\n\n\ndef numerical_transform_stats(train_examples: DataFrame, numeric_cols: list[str]) -> dict:\n    existing = [c for c in numeric_cols if c in train_examples.columns]\n    if not existing:\n        return {}\n\n    agg_exprs = []\n    for col in existing:\n        value = F.col(col).cast("double")\n        agg_exprs.extend(\n            [\n                F.count(value).alias(f"{col}__non_null"),\n                F.mean(value).alias(f"{col}__mean"),\n                F.stddev(value).alias(f"{col}__stddev"),\n                F.min(value).alias(f"{col}__min"),\n                F.percentile_approx(value, [0.01, 0.5, 0.99], 1000).alias(f"{col}__quantiles"),\n                F.max(value).alias(f"{col}__max"),\n            ]\n        )\n\n    row = train_examples.agg(*agg_exprs).first()\n    stats = {}\n    for col in existing:\n        qs = row[f"{col}__quantiles"] or [None, None, None]\n        stats[col] = {\n            "fit_split": "train",\n            "non_null": int(row[f"{col}__non_null"] or 0),\n            "mean": float(row[f"{col}__mean"]) if row[f"{col}__mean"] is not None else None,\n            "stddev": float(row[f"{col}__stddev"]) if row[f"{col}__stddev"] is not None else None,\n            "min": float(row[f"{col}__min"]) if row[f"{col}__min"] is not None else None,\n            "p01": float(qs[0]) if qs[0] is not None else None,\n            "median": float(qs[1]) if qs[1] is not None else None,\n            "p99": float(qs[2]) if qs[2] is not None else None,\n            "max": float(row[f"{col}__max"]) if row[f"{col}__max"] is not None else None,\n        }\n    return stats\n\n\ndef build_feature_catalog(total_rows: int | None = None) -> list[dict]:\n    rows: list[dict] = []\n\n    def add(name: str, source: str, description: str, assignment: str, state: str, pit: str, leakage: str, transform: str, reason: str) -> None:\n        rows.append(\n            {\n                "feature_name": name,\n                "source": source,\n                "semantic_description": description,\n                "tower_assignment": assignment,\n                "static_vs_dynamic": state,\n                "point_in_time_availability": pit,\n                "leakage_risk": leakage,\n                "missingness_or_coverage": "computed in transforms/profile outputs" if total_rows is None else "see transform statistics",\n                "cardinality": "see vocabularies for categorical features",\n                "transformation_needed": transform,\n                "serving_time_feasibility": "feasible if maintained in online/session feature store",\n                "reason": reason,\n            }\n        )\n\n    for c in USER_STATIC_NUMERIC + USER_ANON_NUMERIC:\n        add(c, "silver.users", "static user profile/anonymous attribute", "USER_TOWER", "static snapshot", "known before request", "LOW/MEDIUM", "impute + normalize from train stats", "selected as user-side prior where available")\n    for c in USER_STATIC_CATEGORICAL:\n        add(c, "silver.users", "categorical user activity segment", "USER_TOWER", "static snapshot", "known before request", "LOW/MEDIUM", "train-only vocabulary with OOV", "kept for cold-start user prior")\n    for c in USER_HISTORY_NUMERIC:\n        add(c, "silver.interactions", "strictly prior user/session behavior aggregate", "USER_TOWER", "dynamic point-in-time", "computed with rows before current event only", "LOW", "impute missing cold-start values + normalize", "selected for adaptive short-term and long-term preference")\n    for c in CONTEXT_FEATURES:\n        add(c, "silver.interactions", "request/logging context known at impression time", "USER_TOWER", "request context", "known at request/event time", "LOW/MEDIUM", "categorical or numeric encoding later", "selected as context available to user tower")\n    for c in ITEM_STATIC_NUMERIC:\n        add(c, "silver.videos_basic", "item metadata independent of current user", "ITEM_TOWER", "static/slowly changing", "known before recommendation if metadata exists", "LOW", "impute + normalize from train stats", "selected for item cold-start and embedding precompute")\n    for c in ITEM_STATIC_CATEGORICAL:\n        add(c, "silver.videos_basic", "categorical item metadata", "ITEM_TOWER", "static/slowly changing", "known before recommendation if metadata exists", "LOW", "train-only vocabulary with OOV", "selected for item content signal")\n    for c in ITEM_HISTORY_NUMERIC:\n        add(c, "silver.interactions", "prior item engagement aggregate independent of current user", "ITEM_TOWER", "dynamic point-in-time", "computed with item rows before current event only", "LOW", "impute cold-item values + normalize", "selected as precomputable item popularity/quality signal")\n    for c in ["watch_ratio", "play_time_ms", "duration_ms", *DIRECT_FEEDBACK_COLS]:\n        add(c, "silver.interactions", "current-event outcome/direct feedback", "TARGET_ONLY", "post-event", "available only after current interaction", "HIGH if used as feature", "preserve raw", "target construction only")\n    for c in ["show_cnt", "play_per_show", "like_rate", "comment_rate", "share_rate", "follow_rate"]:\n        add(c, "silver.videos_statistics", "global aggregate snapshot with unclear as-of timestamp", "DROP", "unknown snapshot", "not guaranteed point-in-time", "HIGH/UNKNOWN", "none in v1", "dropped until snapshot semantics are verified or rebuilt historically")\n    add("tag", "silver.videos_basic", "raw multi-value text/tag field", "RESERVED_FOR_RANKER", "static metadata", "known if metadata exists", "LOW", "parse/tokenize later", "reserved because v1 avoids text/multivalue processing")\n    add("user_item_similarity", "future feature store", "cross feature depending on current user and item", "RESERVED_FOR_RANKER", "dynamic cross", "requires both towers", "N/A", "none", "not valid for retrieval tower decomposition")\n    return rows\n\n\ndef quality_checks(examples: DataFrame) -> dict:\n    duplicate_examples = examples.groupBy("example_id").count().where(F.col("count") > 1).count()\n    row = examples.agg(\n        F.sum(\n            (\n                (F.col("history_end_user_event_index") >= F.col("user_event_index"))\n                | (F.col("history_end_session_event_index") >= F.col("session_event_index"))\n            ).cast("long")\n        ).alias("bad_history"),\n        F.sum((F.col("user_hist_events") != F.col("history_end_user_event_index")).cast("long")).alias(\n            "bad_user_history_count"\n        ),\n        F.sum(\n            ((F.col("history_end_user_event_index") == 0) & (F.col("user_hist_events") != 0)).cast("long")\n        ).alias("current_item_in_first_history"),\n        F.sum(F.col("time_ms").isNull().cast("long")).alias("null_time_rows"),\n    ).first()\n    bad_history = int(row["bad_history"] or 0)\n    bad_user_history_count = int(row["bad_user_history_count"] or 0)\n    current_item_in_first_history = int(row["current_item_in_first_history"] or 0)\n    null_time_rows = int(row["null_time_rows"] or 0)\n    return {\n        "duplicate_example_ids": int(duplicate_examples),\n        "bad_history_index_rows": bad_history,\n        "bad_user_history_count_rows": bad_user_history_count,\n        "first_history_contains_rows": current_item_in_first_history,\n        "null_time_rows": null_time_rows,\n        "passed": duplicate_examples == 0 and bad_history == 0 and bad_user_history_count == 0 and current_item_in_first_history == 0 and null_time_rows == 0,\n    }\n\n\ndef write_gold_dataset(args: argparse.Namespace) -> dict:\n    spark = get_spark("kuairand-build-two-tower-gold", reset=True)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    tables = read_silver_tables(spark, args.silver_dir)\n    validate_silver_schema(tables)\n\n    examples, split_meta = build_examples(\n        tables["interactions"],\n        tables["users"],\n        tables["videos_basic"],\n        args.train_quantile,\n        args.validation_quantile,\n        args.weak_watch_ratio_threshold,\n    )\n    storage_level = storage_level_from_env()\n    examples = add_cold_start_flags(examples).persist(storage_level)\n\n    categorical_cols = [c for c in USER_STATIC_CATEGORICAL + ITEM_STATIC_CATEGORICAL if c in examples.columns]\n    numeric_cols = [\n        c\n        for c in USER_STATIC_NUMERIC\n        + USER_ANON_NUMERIC\n        + USER_HISTORY_NUMERIC\n        + CONTEXT_FEATURES\n        + ITEM_STATIC_NUMERIC\n        + ITEM_HISTORY_NUMERIC\n        if c in examples.columns\n    ]\n    train_examples = examples.where(F.col("split") == "train").persist(storage_level)\n\n    vocab_stats = vocabulary_stats(train_examples, categorical_cols)\n    transform_stats = numerical_transform_stats(train_examples, numeric_cols)\n    feature_catalog = build_feature_catalog()\n    checks = quality_checks(examples)\n    if not checks["passed"]:\n        raise AssertionError(f"Two-tower gold quality checks failed: {checks}")\n\n    base_cols = [\n        "example_id",\n        "context_id",\n        "event_id",\n        "user_id",\n        "video_id",\n        "session_id",\n        "user_event_index",\n        "session_event_index",\n        "history_end_user_event_index",\n        "history_end_session_event_index",\n        "as_of_time",\n        "time_ms",\n        "split",\n        "is_warm_user",\n        "is_warm_item",\n        "has_user_history",\n        "has_item_metadata",\n    ]\n    user_cols = [c for c in USER_STATIC_CATEGORICAL + USER_STATIC_NUMERIC + USER_ANON_NUMERIC + USER_HISTORY_NUMERIC + CONTEXT_FEATURES if c in examples.columns]\n    item_cols = [c for c in ITEM_STATIC_CATEGORICAL + ITEM_STATIC_NUMERIC + ITEM_HISTORY_NUMERIC if c in examples.columns]\n    target_cols = [c for c in TARGET_COLS if c in examples.columns]\n    split_output_cols = base_cols\n\n    for split in ["train", "validation", "test"]:\n        write_parquet(examples.where(F.col("split") == split).select(*split_output_cols), args.output_dir / split, args.overwrite)\n        write_parquet(\n            examples.where(F.col("split") == split).select("example_id", "context_id", "user_id", "video_id", "as_of_time", *target_cols),\n            args.output_dir / "targets" / split,\n            args.overwrite,\n        )\n        write_parquet(\n            examples.where(F.col("split") == split).select("example_id", "context_id", "user_id", "session_id", *user_cols),\n            args.output_dir / "user_state" / split,\n            args.overwrite,\n        )\n        write_parquet(\n            examples.where(F.col("split") == split).select("example_id", "video_id", "as_of_time", *item_cols),\n            args.output_dir / "item_features" / "point_in_time" / split,\n            args.overwrite,\n        )\n\n    item_feature_cols = ["video_id", "author_id", *ITEM_STATIC_CATEGORICAL, "upload_date", *[c for c in ITEM_STATIC_NUMERIC if c != "upload_age_days_at_event"]]\n    item_feature_cols = [c for c in item_feature_cols if c in tables["videos_basic"].columns]\n    write_parquet(\n        tables["videos_basic"].select(*item_feature_cols).dropDuplicates(["video_id"]),\n        args.output_dir / "item_features" / "static",\n        args.overwrite,\n    )\n\n    history_cols = [\n        "event_id",\n        "user_id",\n        "video_id",\n        "event_ts",\n        "time_ms",\n        "session_id",\n        "user_event_index",\n        "session_event_index",\n        "target_class",\n        "engagement_strength",\n        *[c for c in DIRECT_FEEDBACK_COLS if c in examples.columns],\n    ]\n    write_parquet(examples.select(*history_cols), args.output_dir / "history" / "events", args.overwrite)\n\n    vocab_dir = args.output_dir / "vocabularies"\n    for col in categorical_cols:\n        write_parquet(build_vocabulary(train_examples, col), vocab_dir / col, args.overwrite)\n    transforms_dir = args.output_dir / "transforms"\n    transforms_dir.mkdir(parents=True, exist_ok=True)\n    write_json({"version": "numeric_transforms_v1", "fit_split": "train", "features": transform_stats}, transforms_dir / "numeric_stats.json")\n    write_json({"version": "categorical_vocabs_v1", "unknown_token": UNKNOWN_TOKEN, "features": vocab_stats}, vocab_dir / "manifest.json")\n    write_json({"version": FEATURE_CATALOG_VERSION, "features": feature_catalog}, args.output_dir / "feature_catalog.json")\n\n    examples_summary = split_counts(examples)\n    manifest = {\n        "gold_dataset_version": args.version,\n        "generated_at": datetime.now(timezone.utc).isoformat(),\n        "source_silver_path": str(args.silver_dir),\n        "output_path": str(args.output_dir),\n        "code_commit": git_commit(),\n        "temporal_split": split_meta,\n        "source_event_summary": split_summary(examples.select("split", "user_id", "video_id", "event_ts")),\n        "example_summary": examples_summary,\n        "feature_catalog_version": FEATURE_CATALOG_VERSION,\n        "target_definition": {\n            "version": TARGET_DEFINITION_VERSION,\n            "strong_positive": "long_view/like/comment/forward/follow on current observed item",\n            "observed_negative": f"not strong positive and (is_hate=1 or clicked watch_ratio <= {args.weak_watch_ratio_threshold})",\n            "ambiguous": "observed event without strong positive or reliable observed negative evidence",\n            "unknown_policy": "unobserved user-item pairs are not materialized and are never assumed negative",\n        },\n        "history_definition": {\n            "version": HISTORY_DEFINITION_VERSION,\n            "session_gap_minutes": 30,\n            "history_reference": "Each example stores history_end_user_event_index and history_end_session_event_index; history/events stores chronological events. The current event is excluded by construction.",\n        },\n        "feature_groups": {\n            "user_tower": user_cols,\n            "item_tower": item_cols,\n            "target_only": target_cols,\n            "reserved_for_ranker": ["tag", "user_item_similarity", "future ALS/user-item cross scores"],\n            "dropped": ["videos_statistics global counters/rates with unclear snapshot time", "current-event watch/play outcomes as features"],\n        },\n        "cold_start": cold_start_stats(examples),\n        "vocabularies": {"version": "categorical_vocabs_v1", "fit_split": "train", "features": vocab_stats},\n        "numerical_transforms": {"version": "numeric_transforms_v1", "fit_split": "train", "features": list(transform_stats)},\n        "quality_checks": checks,\n        "artifact_paths": {\n            "train": str(args.output_dir / "train"),\n            "validation": str(args.output_dir / "validation"),\n            "test": str(args.output_dir / "test"),\n            "user_state": str(args.output_dir / "user_state"),\n            "item_features_static": str(args.output_dir / "item_features" / "static"),\n            "item_features_point_in_time": str(args.output_dir / "item_features" / "point_in_time"),\n            "history": str(args.output_dir / "history" / "events"),\n            "targets": str(args.output_dir / "targets"),\n            "vocabularies": str(args.output_dir / "vocabularies"),\n            "transforms": str(args.output_dir / "transforms"),\n            "feature_catalog": str(args.output_dir / "feature_catalog.json"),\n            "manifest": str(args.output_dir / "manifest.json"),\n        },\n    }\n    write_json(manifest, args.output_dir / "manifest.json")\n    print(json.dumps(manifest, indent=2, sort_keys=True))\n    spark.stop()\n    return manifest\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Build point-in-time KuaiRand Gold data for a future Two-Tower retrieval model.",\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n    )\n    parser.add_argument("--silver-dir", type=Path, default=Path("data/silver/kuairand"))\n    parser.add_argument("--output-dir", type=Path, default=Path("data/gold/two_tower/v1"))\n    parser.add_argument("--version", default=DATASET_VERSION)\n    parser.add_argument("--train-quantile", type=float, default=0.70)\n    parser.add_argument("--validation-quantile", type=float, default=0.85)\n    parser.add_argument("--weak-watch-ratio-threshold", type=float, default=0.20)\n    parser.add_argument("--overwrite", action="store_true")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    write_gold_dataset(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n')

if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))

print('Runtime package written to', PACKAGE_ROOT)


## Build Gold Dataset

This is the long-running cell. If it succeeds, it writes the complete local Gold artifact before copying it to Drive.

In [ ]:
import argparse
import json

from recommender.gold.two_tower import write_gold_dataset

if LOCAL_OUTPUT_DIR.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIR)

args = argparse.Namespace(
    silver_dir=LOCAL_SILVER_DIR,
    output_dir=LOCAL_OUTPUT_DIR,
    version='two_tower_v1_colab',
    train_quantile=0.70,
    validation_quantile=0.85,
    weak_watch_ratio_threshold=0.20,
    overwrite=True,
)
manifest = write_gold_dataset(args)
print('Build finished. Manifest keys:', sorted(manifest.keys()))


## Validate Local Artifact

In [ ]:
from pathlib import Path

expected_success = [
    'train/_SUCCESS',
    'validation/_SUCCESS',
    'test/_SUCCESS',
    'targets/train/_SUCCESS',
    'targets/validation/_SUCCESS',
    'targets/test/_SUCCESS',
    'user_state/train/_SUCCESS',
    'user_state/validation/_SUCCESS',
    'user_state/test/_SUCCESS',
    'item_features/static/_SUCCESS',
    'item_features/point_in_time/train/_SUCCESS',
    'item_features/point_in_time/validation/_SUCCESS',
    'item_features/point_in_time/test/_SUCCESS',
    'history/events/_SUCCESS',
]
missing_success = [p for p in expected_success if not (LOCAL_OUTPUT_DIR / p).exists()]
if missing_success:
    raise RuntimeError(f'Missing completed outputs: {missing_success}')

for file_name in ['manifest.json', 'feature_catalog.json', 'vocabularies/manifest.json', 'transforms/numeric_stats.json']:
    if not (LOCAL_OUTPUT_DIR / file_name).exists():
        raise RuntimeError(f'Missing metadata file: {file_name}')

print('All expected _SUCCESS markers and metadata files exist.')
print('Local output size GB:', sum(p.stat().st_size for p in LOCAL_OUTPUT_DIR.rglob('*') if p.is_file()) / 1024**3)


## Copy Gold Artifact Back to Drive

In [ ]:
if DRIVE_OUTPUT_DIR.exists():
    shutil.rmtree(DRIVE_OUTPUT_DIR)
DRIVE_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR)
print('Copied Gold artifact to:', DRIVE_OUTPUT_DIR)
print('Drive output size GB:', sum(p.stat().st_size for p in DRIVE_OUTPUT_DIR.rglob('*') if p.is_file()) / 1024**3)


## Final Summary

In [ ]:
manifest = json.loads((DRIVE_OUTPUT_DIR / 'manifest.json').read_text())
summary = {
    'gold_dataset_version': manifest['gold_dataset_version'],
    'temporal_split': manifest['temporal_split'],
    'example_summary': manifest['example_summary'],
    'cold_start': manifest['cold_start'],
    'artifact_paths': manifest['artifact_paths'],
}
print(json.dumps(summary, indent=2)[:12000])
